# 4장 Text Classification — SageMaker AI Studio(JupyterLab)용

<!-- origin-notice -->
> **원본 노트북:** [chapter04/Chapter 4 - Text Classification.ipynb](https://github.com/HandsOnLLM/Hands-On-Large-Language-Models/blob/main/chapter04/Chapter%204%20-%20Text%20Classification.ipynb)  
> **원본 저장소:** [HandsOnLLM/Hands-On-Large-Language-Models](https://github.com/HandsOnLLM/Hands-On-Large-Language-Models) (Apache License 2.0, © 2024 Jay Alammar & Maarten Grootendorst)  
> 이 노트북은 원본을 SageMaker AI Studio에서 실행되도록 수정하고 한국어 설명·보충 실험을 추가한 것입니다. 변경 내용은 노트북 안의 "책과 다를 수 있는 점"에 적혀 있습니다.

원본: HandsOnLLM/Hands-On-Large-Language-Models `chapter04` 노트북(Apache-2.0)을 수정·보충했습니다. 권장 인스턴스 `ml.g4dn.xlarge`(T4 16GB), 스토리지 30GB 이상. 이 장의 모델은 모두 1GB 미만이라 GPU 메모리 여유가 큽니다.

영화 리뷰 한 문장을 긍정/부정으로 나누는 같은 문제를 **네 가지 방식**으로 풀고 점수를 비교합니다.

| 절 | 하는 일 | 책 위치(PDF 쪽) |
|---|---|---|
| 1 | 데이터: Rotten Tomatoes 영화 리뷰(긍정/부정 반반) | 161–163 |
| 2 | 표현 모델 ① 감성 분류용으로 이미 학습된 모델을 그대로 쓰기 (+ "중립" 클래스 분석) | 163–171 |
| 3 | 표현 모델 ② 임베딩 + 가벼운 분류기(로지스틱 회귀) (+ 라벨 수에 따른 성능 곡선) | 171–176 |
| 4 | 표현 모델 ③ 라벨 없이: 임베딩 제로샷 분류 (+ 라벨 문구 바꾸기, 주행 로그 분류 예) | 176–180 |
| 5 | 생성 모델 ① Flan-T5에게 "긍정이야 부정이야?" 묻기 | 181–186 |
| 6 | 생성 모델 ② 상용 LLM API로 분류 (책: OpenAI, 책 밖 대안: Amazon Bedrock) | 186–190 |
| 7 | 점수 비교표: 책 값과 내 실행 결과 | — |

`(+ …)` 표시는 책에 없는 보충 실험입니다. 셀은 위에서부터 차례로 실행하세요.

**용어.** 이 장에서 **표현 모델(representation model)**은 BERT 계열처럼 텍스트를 벡터로 바꾸는 모델, **생성 모델(generative model)**은 GPT·T5처럼 텍스트를 만들어 내는 모델입니다(1장 그림의 청록색/분홍색 구분).

## 0. 환경 점검과 패키지 설치

이미지에 없는 패키지만 설치합니다. `openai`는 6절에서 키가 있을 때만 쓰므로 여기서 설치하지 않습니다.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h ~ | tail -1
import importlib, subprocess, sys
need = {"datasets": "datasets", "sentence_transformers": "sentence-transformers", "sklearn": "scikit-learn"}
missing = [pkg for mod, pkg in need.items() if importlib.util.find_spec(mod) is None]
if missing:
    print("설치:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
import torch, transformers, gc
import numpy as np, pandas as pd
from tqdm.auto import tqdm
print("torch", torch.__version__, "| CUDA", torch.cuda.is_available(), "| transformers", transformers.__version__)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

## 1. 데이터: 영화 리뷰의 감성 (PDF 161–163)

학습 8,530개, 검증 1,066개, 테스트 1,066개이고 라벨은 0(부정)/1(긍정)입니다.

> **책과 다를 수 있는 점.** 책 코드의 `load_dataset("rotten_tomatoes")`는 최신 `datasets`/`huggingface_hub`에서 "Repository id must be 'namespace/name'" 오류가 납니다. 같은 데이터의 정식 저장소 이름 `cornell-movie-review-data/rotten_tomatoes`를 씁니다.

In [ ]:
from datasets import load_dataset
data = load_dataset("cornell-movie-review-data/rotten_tomatoes")
data

In [ ]:
data["train"][0, -1]

점수는 책과 같이 `classification_report`의 **가중 평균 F1**(weighted avg 행, f1-score 열)로 비교합니다. F1은 정밀도(precision, 긍정이라고 한 것 중 실제 긍정 비율)와 재현율(recall, 실제 긍정 중 찾아낸 비율)의 조화평균입니다. 아래 함수는 책 함수에 **결과를 `results`에 모아 두는 기능**만 더했습니다(7절 비교표용).

In [ ]:
from sklearn.metrics import classification_report, f1_score
results = {}

def evaluate_performance(y_true, y_pred, name=None):
    """Create and print the classification report"""
    print(classification_report(y_true, y_pred, target_names=["Negative Review", "Positive Review"]))
    if name:
        results[name] = round(f1_score(y_true, y_pred, average="weighted"), 2)

## 2. 감성 분류용으로 학습된 모델 그대로 쓰기 (PDF 163–171)

`cardiffnlp/twitter-roberta-base-sentiment-latest`는 RoBERTa(BERT 계열)를 트윗 감성 분류로 미세조정한 모델입니다. 출력 라벨이 negative / neutral / positive **세 개**라서, 책은 neutral을 버리고 negative와 positive 점수만 비교합니다. 영화 리뷰가 아니라 트윗으로 학습했다는 점도 기억해 두세요(도메인 차이).

> **책과 다를 수 있는 점.** 책 코드의 `return_all_scores=True`는 transformers 5.x에서 더 이상 동작하지 않아 1등 라벨 하나만 돌려줍니다. 그러면 `output[2]`에서 오류가 납니다. `top_k=None`을 쓰면 세 점수가 모두 나오지만 **점수 순으로 정렬**되므로, 책처럼 `output[0]`/`output[2]` 위치로 꺼내면 틀립니다. 아래 코드는 라벨 이름으로 꺼냅니다(4.x에서도 동작).

In [ ]:
from transformers import pipeline
from transformers.pipelines.pt_utils import KeyDataset

model_path = "cardiffnlp/twitter-roberta-base-sentiment-latest"
pipe = pipeline(model=model_path, tokenizer=model_path, top_k=None, device=DEVICE)

In [ ]:
y_pred, all_scores = [], []
for output in tqdm(pipe(KeyDataset(data["test"], "text"), batch_size=32), total=len(data["test"])):
    scores = {d["label"]: d["score"] for d in output}       # 위치가 아니라 라벨 이름으로
    all_scores.append(scores)
    y_pred.append(int(np.argmax([scores["negative"], scores["positive"]])))

In [ ]:
evaluate_performance(data["test"]["label"], y_pred, name="task-specific (RoBERTa sentiment)")

### (+ 보충) 버려진 "중립"은 얼마나 많았나

neutral 점수가 가장 높았던 리뷰는 모델이 사실상 판단을 보류한 경우입니다. 이 리뷰들에서 정확도가 얼마나 떨어지는지 봅니다. 미리 정해진 라벨 체계가 실제 문제와 맞지 않을 때 생기는 손실입니다.

In [ ]:
sc = pd.DataFrame(all_scores)
sc["top"] = sc.idxmax(axis=1)
sc["correct"] = np.array(y_pred) == np.array(data["test"]["label"])
summary = sc.groupby("top")["correct"].agg(count="size", accuracy="mean").round(3)
print(summary)
print("\n중립이 1등이었던 리뷰 예시:")
for t in np.array(data["test"]["text"])[(sc["top"] == "neutral").values][:3]:
    print(" -", t)

In [ ]:
del pipe; gc.collect(); torch.cuda.empty_cache()

## 3. 임베딩 + 분류기 (PDF 171–176)

미세조정 없이 2장의 문장 임베딩 모델(`all-mpnet-base-v2`, 768차원)로 리뷰를 벡터로 바꾸고, 그 위에 로지스틱 회귀(logistic regression)만 학습합니다. 임베딩 모델은 고정(frozen)이고 학습하는 것은 768개 가중치 + 편향뿐이라 CPU로도 몇 초면 끝납니다.

In [ ]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2", device=DEVICE)
train_embeddings = model.encode(data["train"]["text"], show_progress_bar=True)
test_embeddings = model.encode(data["test"]["text"], show_progress_bar=True)
train_embeddings.shape

In [ ]:
from sklearn.linear_model import LogisticRegression
clf = LogisticRegression(random_state=42)
clf.fit(train_embeddings, data["train"]["label"])
y_pred = clf.predict(test_embeddings)
evaluate_performance(data["test"]["label"], y_pred, name="embeddings + logistic regression")

**책의 팁: 분류기 없이.** 클래스마다 학습 임베딩의 평균을 내 "대표 벡터"를 만들고, 테스트 리뷰를 코사인 유사도가 더 높은 쪽으로 분류합니다. 학습이라고 할 것이 평균 계산뿐인데도 점수가 크게 떨어지지 않습니다.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
df = pd.DataFrame(np.hstack([train_embeddings, np.array(data["train"]["label"]).reshape(-1, 1)]))
averaged_target_embeddings = df.groupby(768).mean().values
sim_matrix = cosine_similarity(test_embeddings, averaged_target_embeddings)
y_pred = np.argmax(sim_matrix, axis=1)
evaluate_performance(data["test"]["label"], y_pred, name="class-average embeddings (no classifier)")

### (+ 보충) 라벨이 몇 개 있어야 할까

라벨링은 비용입니다. 학습 데이터를 8개, 32개, … 로 줄여 가며 로지스틱 회귀의 F1이 어떻게 변하는지 봅니다(크기마다 무작위 표본 3번 평균). 좋은 임베딩 위에서는 수백 개만으로도 전체 데이터에 가까운 성능이 나오는 경우가 많습니다.

In [ ]:
rng = np.random.default_rng(0)
y_train = np.array(data["train"]["label"])
sizes = [n for n in [8, 32, 128, 512, 2048] if n < len(y_train)] + [len(y_train)]
rows = []
for n in sizes:
    f1s = []
    for _ in range(3 if n < len(y_train) else 1):
        while True:                                   # 두 클래스가 모두 들어가게 뽑기
            idx = rng.choice(len(y_train), n, replace=False)
            if len(set(y_train[idx])) == 2: break
        m = LogisticRegression(random_state=42, max_iter=1000).fit(train_embeddings[idx], y_train[idx])
        f1s.append(f1_score(data["test"]["label"], m.predict(test_embeddings), average="weighted"))
    rows.append({"labeled_examples": n, "weighted_f1": round(float(np.mean(f1s)), 3)})
curve = pd.DataFrame(rows).set_index("labeled_examples")
curve

In [ ]:
ax = curve.plot(marker="o", logx=True, legend=False, figsize=(6, 3.5), title="F1 vs. number of labeled examples")
ax.set_xlabel("labeled training examples (log)"); ax.set_ylabel("weighted F1"); ax.grid(alpha=.3)

## 4. 라벨이 없을 때: 임베딩 제로샷 분류 (PDF 176–180)

라벨 데이터가 하나도 없으면 **라벨 자체를 문장으로 써서** 임베딩합니다("A negative review", "A positive review"). 리뷰 임베딩과 더 가까운 라벨 문장을 답으로 고릅니다.

In [ ]:
label_embeddings = model.encode(["A negative review", "A positive review"])
sim_matrix = cosine_similarity(test_embeddings, label_embeddings)
y_pred = np.argmax(sim_matrix, axis=1)
evaluate_performance(data["test"]["label"], y_pred, name="zero-shot (label embeddings)")

### (+ 보충) 라벨 문구에 따라 점수가 얼마나 달라지나

책의 팁("A very negative movie review"를 써 보라)을 표로 확장했습니다. 제로샷 분류에서는 라벨 문장을 쓰는 일이 사실상 **프롬프트 엔지니어링**입니다(6장).

In [ ]:
variants = {
    "A negative review / A positive review": ["A negative review", "A positive review"],
    "A very negative movie review / A very positive movie review": ["A very negative movie review", "A very positive movie review"],
    "negative / positive": ["negative", "positive"],
    "I hated this movie / I loved this movie": ["I hated this movie", "I loved this movie"],
}
rows = []
for name, labels in variants.items():
    pred = cosine_similarity(test_embeddings, model.encode(labels)).argmax(1)
    rows.append({"label descriptions": name, "weighted_f1": round(f1_score(data["test"]["label"], pred, average="weighted"), 3)})
pd.DataFrame(rows).set_index("label descriptions")

**주행 로그에 적용해 보기(책 밖).** 같은 방법으로 라벨 없는 주행 이벤트 메모를 범주로 나눌 수 있습니다. 라벨링 전에 데이터를 대략 분류해 두거나, 특정 유형의 장면을 먼저 찾을 때 쓰는 방식입니다. 아래 문장과 범주는 설명용으로 만든 예시입니다.

범주 문장을 두 벌 준비했습니다. **추상적인 범주명**("a pedestrian or cyclist event")과, 로그에 실제로 나올 법한 말로 쓴 **구체적인 설명**("a person, child, or cyclist moving into the road")입니다. 제작 시 CPU 실행에서는 추상적인 쪽이 6건 중 2건(아이, 자전거)을 틀렸고 구체적인 쪽은 모두 맞혔습니다. 위 영화 리뷰 표와 같은 교훈입니다.

In [ ]:
logs = ["Child ran onto the road chasing a ball near the school zone.",
        "Heavy rain made the lane markings almost invisible on the highway.",
        "A truck merged into our lane without signaling at the on-ramp.",
        "The signal at the intersection was stuck on flashing yellow.",
        "Dense fog reduced visibility to about thirty meters.",
        "A cyclist swerved around a parked car into the travel lane."]
expected = ["pedestrian", "weather", "cut-in", "signal", "weather", "pedestrian"]
short = ["pedestrian", "weather", "cut-in", "signal"]
category_sets = {
    "abstract": ["a pedestrian or cyclist event", "bad weather or low visibility",
                 "another vehicle cutting in or merging", "a traffic light or road sign issue"],
    "concrete": ["a person, child, or cyclist moving into the road", "rain, fog, or snow reducing visibility",
                 "a car or truck cutting into our lane", "a broken or confusing traffic signal"],
}
log_emb = model.encode(logs)
out = pd.DataFrame({"log": logs, "expected": expected})
for name, cats in category_sets.items():
    out[name] = [short[i] for i in cosine_similarity(log_emb, model.encode(cats)).argmax(1)]
print({name: f"{(out[name] == out['expected']).sum()}/{len(out)} correct" for name in category_sets})
out

In [ ]:
del model; gc.collect(); torch.cuda.empty_cache()

## 5. 생성 모델로 분류: Flan-T5 (PDF 181–186)

T5는 인코더-디코더(encoder-decoder) 구조의 생성 모델이고, Flan-T5는 여러 작업을 지시문 형태로 미세조정한 버전입니다. 리뷰 앞에 질문을 붙여 넣으면 모델이 "positive"/"negative"라는 **단어를 생성**합니다. 분류를 텍스트 생성 문제로 바꾼 것입니다.

> **책과 다를 수 있는 점.** 책 코드의 `pipeline("text2text-generation", ...)`은 transformers 5.x에서 제거되어 "Unknown task" 오류가 납니다. 아래는 같은 일을 `AutoModelForSeq2SeqLM.generate`로 직접 하는 코드이고, 4.x에서도 동작합니다. 32개씩 묶어(batch) 처리해서 책보다 빠릅니다.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
t5_tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
t5_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small").to(DEVICE).eval()

prompt = "Is the following sentence positive or negative? "
data = data.map(lambda example: {"t5": prompt + example["text"]})
data

In [ ]:
y_pred, raw = [], []
texts = data["test"]["t5"]
for i in tqdm(range(0, len(texts), 32)):
    enc = t5_tokenizer(texts[i:i+32], return_tensors="pt", padding=True, truncation=True).to(DEVICE)
    with torch.no_grad():
        out = t5_model.generate(**enc, max_new_tokens=5)
    for text in t5_tokenizer.batch_decode(out, skip_special_tokens=True):
        raw.append(text)
        y_pred.append(0 if text == "negative" else 1)
print("생성된 답의 종류:", pd.Series(raw).value_counts().to_dict())
evaluate_performance(data["test"]["label"], y_pred, name="Flan-T5-small (generative)")

**관찰 포인트.** 책 코드는 `"negative"`가 아니면 모두 긍정(1)으로 처리합니다. 위 "생성된 답의 종류"에 positive/negative 말고 다른 답이 섞여 있다면 그만큼 긍정 쪽으로 치우칩니다. 생성 모델로 분류할 때는 **출력 형식을 검증**하는 단계가 필요합니다(6장 Output Verification).

In [ ]:
del t5_model; gc.collect(); torch.cuda.empty_cache()

## 6. 상용 LLM API로 분류 (PDF 186–190)

책은 OpenAI의 `gpt-3.5-turbo-0125`로 테스트 1,066건을 분류해 F1 0.91을 얻었습니다. 유료이고 키가 필요하므로 아래 셀은 `OPENAI_API_KEY` 환경 변수가 없으면 건너뜁니다. 책에 쓰인 모델 이름은 지금은 제공되지 않을 수 있으니 OpenAI 문서에서 현재 모델을 확인하세요.

프롬프트의 `[DOCUMENT]` 자리에 리뷰를 넣고 "긍정이면 1, 부정이면 0만 답하라"고 출력 형식을 정해 줍니다.

In [ ]:
import os
prompt = """Predict whether the following document is a positive or negative movie review:

[DOCUMENT]

If it is positive return 1 and if it is negative return 0. Do not give any other answers.
"""
N_EVAL = 100        # 비용을 아끼려고 테스트 앞 100건만. 전체는 len(data["test"])

def parse_label(text):
    """응답에서 0/1을 꺼낸다. 형식이 틀리면 -1"""
    t = text.strip()
    return int(t[0]) if t[:1] in ("0", "1") else -1

def report(preds, name):
    y_true = data["test"]["label"][:len(preds)]
    bad = sum(p == -1 for p in preds)
    print(f"형식 오류 {bad}건(부정으로 처리)")
    evaluate_performance(y_true, [max(p, 0) for p in preds], name=name)

In [ ]:
if os.environ.get("OPENAI_API_KEY"):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "openai"])
    import openai
    client = openai.OpenAI()
    OPENAI_MODEL = "gpt-3.5-turbo-0125"     # 책의 모델. 제공 종료되었으면 현재 모델로 바꾸세요

    def chatgpt_generation(prompt, document, model=OPENAI_MODEL):
        messages = [{"role": "system", "content": "You are a helpful assistant."},
                    {"role": "user", "content": prompt.replace("[DOCUMENT]", document)}]
        r = client.chat.completions.create(messages=messages, model=model, temperature=0)
        return r.choices[0].message.content

    print(chatgpt_generation(prompt, "unpretentious , charming , quirky , original"))
    preds = [parse_label(chatgpt_generation(prompt, d)) for d in tqdm(data["test"]["text"][:N_EVAL])]
    report(preds, f"OpenAI {OPENAI_MODEL} (first {N_EVAL})")
else:
    print("OPENAI_API_KEY가 없어 건너뜁니다.")

### (책 밖) Amazon Bedrock으로 같은 실험

SageMaker Studio의 실행 역할(execution role)에 Bedrock 호출 권한이 있고, 사용할 모델의 접근이 활성화되어 있어야 합니다. `BEDROCK_MODEL_ID`에 Bedrock 콘솔에서 확인한 모델 ID(또는 추론 프로파일 ID)를 넣으면 실행됩니다. 비워 두면 건너뜁니다. 호출 방식은 모델 종류와 관계없이 같은 Converse API입니다.

In [ ]:
BEDROCK_MODEL_ID = ""      # 예: Bedrock 콘솔 > Model catalog에서 복사한 ID
BEDROCK_REGION = None      # None이면 Studio 기본 리전

if BEDROCK_MODEL_ID:
    import boto3
    brt = boto3.client("bedrock-runtime", region_name=BEDROCK_REGION)

    def bedrock_generation(prompt, document):
        r = brt.converse(
            modelId=BEDROCK_MODEL_ID,
            system=[{"text": "You are a helpful assistant."}],
            messages=[{"role": "user", "content": [{"text": prompt.replace("[DOCUMENT]", document)}]}],
            inferenceConfig={"temperature": 0, "maxTokens": 5},
        )
        return r["output"]["message"]["content"][0]["text"]

    print(bedrock_generation(prompt, "unpretentious , charming , quirky , original"))
    preds = [parse_label(bedrock_generation(prompt, d)) for d in tqdm(data["test"]["text"][:N_EVAL])]
    report(preds, f"Bedrock {BEDROCK_MODEL_ID} (first {N_EVAL})")
else:
    print("BEDROCK_MODEL_ID가 비어 있어 건너뜁니다.")

## 7. 점수 비교: 책 값과 내 결과

책 값은 PDF 본문과 공식 노트북 출력의 가중 평균 F1입니다. 내 결과가 책과 조금 다른 것은 정상입니다(라이브러리 버전, GPU 종류, 6절은 표본 수 차이).

In [ ]:
book = {"task-specific (RoBERTa sentiment)": 0.80,
        "embeddings + logistic regression": 0.85,
        "class-average embeddings (no classifier)": 0.84,
        "zero-shot (label embeddings)": 0.78,
        "Flan-T5-small (generative)": 0.84}
names = list(book) + [k for k in results if k not in book]
pd.DataFrame({"book_f1": [book.get(k, 0.91 if "OpenAI" in k else None) for k in names],
              "my_f1": [results.get(k) for k in names]}, index=names)

| 방식 | 라벨 데이터 | 학습 | 비용·속도 |
|---|---|---|---|
| 감성 분류 전용 모델 | 필요 없음(이미 학습됨) | 없음 | 빠름. 라벨 체계·도메인이 맞아야 함 |
| 임베딩 + 분류기 | 필요(수백 개면 상당 부분 확보) | 분류기만, 몇 초 | 빠름. 가장 실용적인 기준선 |
| 임베딩 제로샷 | 없음 | 없음 | 빠름. 라벨 문구에 민감 |
| 생성 모델(Flan-T5, 상용 LLM) | 없음 | 없음 | 느리고 API 비용. 출력 형식 검증 필요, 성능 상한은 높음 |

**자율주행·Agent로 이어지는 점(책 밖).** 주행 데이터 파이프라인에서 "이 장면이 어떤 유형인가"를 붙이는 일이 대표적인 분류 문제입니다. 3절의 곡선처럼 좋은 임베딩 위에서는 적은 라벨로도 분류기를 만들 수 있고, 4절처럼 라벨 없이 범주 문장만으로 장면을 먼저 걸러 낼 수 있습니다. 비전 쪽에서는 CLIP 같은 이미지-텍스트 임베딩으로 같은 방법을 씁니다(9장). Agent에서는 사용자 요청을 어느 도구나 하위 Agent로 보낼지 정하는 **라우팅**이 분류 문제이고, 5·6절처럼 생성 모델로 풀 때는 출력 형식 검증이 필요합니다.

**끝나면** Studio에서 space를 **Stop** 하세요.